Setup

Load the API key and relevant Python libaries.

In this course, we've provided some code that loads the OpenAI API key for you.

In [1]:
!pip install groq
from google.colab import userdata
from groq import Groq

client = Groq(
    api_key=userdata.get("GROQ_API_KEY")
)


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 143.7/143.7 kB 2.1 MB/s eta 0:00:00


In [2]:
from openai import OpenAI
from google.colab import userdata

client = OpenAI(
    api_key=userdata.get("GROQ_API_KEY"),
    base_url="https://api.groq.com/openai/v1"
)

def get_completion_from_messages(
    messages,
    model="llama-3.3-70b-versatile",
    temperature=0,
    max_tokens=500
):
    response = client.chat.completions.create(
        model=model,
        messages=messages,
        temperature=temperature,
        max_tokens=max_tokens
    )
    return response.choices[0].message.content

Chain-of-Thought Prompting

In [3]:
delimiter = "####"
system_message = f"""
Follow these steps to answer the customer queries.
The customer query will be delimited with four hashtags,\
i.e. {delimiter}.

Step 1:{delimiter} First decide whether the user is \
asking a question about a specific product or products. \
Product cateogry doesn't count.

Step 2:{delimiter} If the user is asking about \
specific products, identify whether \
the products are in the following list.
All available products:
1. Product: TechPro Ultrabook
   Category: Computers and Laptops
   Brand: TechPro
   Model Number: TP-UB100
   Warranty: 1 year
   Rating: 4.5
   Features: 13.3-inch display, 8GB RAM, 256GB SSD, Intel Core i5 processor
   Description: A sleek and lightweight ultrabook for everyday use.
   Price: $799.99

2. Product: BlueWave Gaming Laptop
   Category: Computers and Laptops
   Brand: BlueWave
   Model Number: BW-GL200
   Warranty: 2 years
   Rating: 4.7
   Features: 15.6-inch display, 16GB RAM, 512GB SSD, NVIDIA GeForce RTX 3060
   Description: A high-performance gaming laptop for an immersive experience.
   Price: $1199.99

3. Product: PowerLite Convertible
   Category: Computers and Laptops
   Brand: PowerLite
   Model Number: PL-CV300
   Warranty: 1 year
   Rating: 4.3
   Features: 14-inch touchscreen, 8GB RAM, 256GB SSD, 360-degree hinge
   Description: A versatile convertible laptop with a responsive touchscreen.
   Price: $699.99

4. Product: TechPro Desktop
   Category: Computers and Laptops
   Brand: TechPro
   Model Number: TP-DT500
   Warranty: 1 year
   Rating: 4.4
   Features: Intel Core i7 processor, 16GB RAM, 1TB HDD, NVIDIA GeForce GTX 1660
   Description: A powerful desktop computer for work and play.
   Price: $999.99

5. Product: BlueWave Chromebook
   Category: Computers and Laptops
   Brand: BlueWave
   Model Number: BW-CB100
   Warranty: 1 year
   Rating: 4.1
   Features: 11.6-inch display, 4GB RAM, 32GB eMMC, Chrome OS
   Description: A compact and affordable Chromebook for everyday tasks.
   Price: $249.99

Step 3:{delimiter} If the message contains products \
in the list above, list any assumptions that the \
user is making in their \
message e.g. that Laptop X is bigger than \
Laptop Y, or that Laptop Z has a 2 year warranty.

Step 4:{delimiter}: If the user made any assumptions, \
figure out whether the assumption is true based on your \
product information.

Step 5:{delimiter}: First, politely correct the \
customer's incorrect assumptions if applicable. \
Only mention or reference products in the list of \
5 available products, as these are the only 5 \
products that the store sells. \
Answer the customer in a friendly tone.

Use the following format:
Step 1:{delimiter} <step 1 reasoning>
Step 2:{delimiter} <step 2 reasoning>
Step 3:{delimiter} <step 3 reasoning>
Step 4:{delimiter} <step 4 reasoning>
Response to user:{delimiter} <response to customer>

Make sure to include {delimiter} to separate every step.
"""

In [4]:
user_message = f"""
by how much is the BlueWave Chromebook more expensive \
than the TechPro Desktop"""

messages =  [
{'role':'system',
 'content': system_message},
{'role':'user',
 'content': f"{delimiter}{user_message}{delimiter}"},
]

response = get_completion_from_messages(messages)
print(response)

Step 1:#### The user is asking a question about the price comparison between two specific products, the BlueWave Chromebook and the TechPro Desktop, which suggests they are inquiring about specific products.

Step 2:#### The products mentioned, BlueWave Chromebook and TechPro Desktop, are both in the list of available products. The BlueWave Chromebook is priced at $249.99 and the TechPro Desktop is priced at $999.99.

Step 3:#### The user is assuming that the BlueWave Chromebook is more expensive than the TechPro Desktop, as they are asking by how much the BlueWave Chromebook is more expensive.

Step 4:#### This assumption is incorrect. According to the product information, the TechPro Desktop ($999.99) is actually more expensive than the BlueWave Chromebook ($249.99).

Response to user:#### I think there might be a misunderstanding. The BlueWave Chromebook is actually less expensive than the TechPro Desktop, not more. The BlueWave Chromebook is priced at $249.99, while the TechPro Des

What is "Process Inputs: Chain of Thought Reasoning"?

Chain of Thought (CoT) Reasoning is a technique where an LLM solves a problem by breaking it into logical intermediate steps before producing the final answer. It is most useful for tasks that require reasoning, planning, or multiple decisions rather than simple fact retrieval.

Think of it like a human solving a math problem or planning a trip: instead of jumping to the answer, they work through the problem step by step.

===

User Input
     │
     ▼
Understand the Request
     │
     ▼
Break the Problem into Smaller Parts
     │
     ▼
Reason Through Each Part
     │
     ▼
Combine the Results
     │
     ▼
Generate Final Answer


====

Example 1: Simple Question
User Input
How many days are there in 3 weeks?
Reasoning Process
1 week = 7 days

3 weeks = 3 × 7

= 21 days
Final Answer
21 days

====

Example 2: Customer Support
User Input
I forgot my password and can't log in.
Reasoning
Step 1:
Identify the issue.

↓

Password problem.

Step 2:
Find the category.

↓

Account Management

Step 3:
Find the subcategory.

↓

Password Reset

Step 4:
Generate the response.
Output
{
  "primary": "Account Management",
  "secondary": "Password reset"
}

====



Example 3: Restaurant Recommendation
User Input
Recommend a vegetarian restaurant near me.
Reasoning
Understand request

↓

Need restaurant

↓

Must be vegetarian

↓

Near user's location

↓

Return recommendation
Example 4: Math
User Input
A pen costs ₹25.

Buy 4 pens.

How much?
Reasoning
Price of one pen = 25

Quantity = 4

25 × 4

= 100
Answer
₹100
Example 5: Travel Planning
User Input
I have two days in Chennai.

Suggest an itinerary.
Reasoning
Need itinerary

↓

Day 1 attractions

↓

Day 2 attractions

↓

Food suggestions

↓

Transportation

↓

Final itinerary

Example 4: Math
User Input
A pen costs ₹25.

Buy 4 pens.

How much?
Reasoning
Price of one pen = 25

Quantity = 4

25 × 4

= 100
Answer
₹100
Example 5: Travel Planning
User Input
I have two days in Chennai.

Suggest an itinerary.
Reasoning
Need itinerary

↓

Day 1 attractions

↓

Day 2 attractions

↓

Food suggestions

↓

Transportation

↓

Final itinerary
Why is Chain of Thought Useful?

| Without Chain of Thought    | With Chain of Thought            |
| --------------------------- | -------------------------------- |
| Direct answer               | Breaks problem into steps        |
| May miss details            | Considers intermediate reasoning |
| Better for simple questions | Better for complex reasoning     |
| Less transparent            | Structured reasoning process     |


====

| Domain            | How Chain of Thought Helps                           |
| ----------------- | ---------------------------------------------------- |
| Customer Support  | Classify issues and decide next actions              |
| Medical AI        | Analyze symptoms before suggesting possibilities     |
| Finance           | Calculate loans, taxes, and investments step by step |
| Education         | Solve math and science problems logically            |
| Programming       | Break coding tasks into smaller steps                |
| Business Analysis | Analyze requirements before proposing solutions      |
| AI Agents         | Plan multi-step tasks before executing them          |
| Robotics          | Decide actions in sequence                           |


In [5]:
user_message = f"""
do you sell tvs"""
messages =  [
{'role':'system',
 'content': system_message},
{'role':'user',
 'content': f"{delimiter}{user_message}{delimiter}"},
]
response = get_completion_from_messages(messages)
print(response)

Step 1:#### The user is asking a general question about the products sold, but not specifically about a product or products.

Step 2:#### The user is not asking about any specific products from the list, but rather inquiring about a product category (TVs) that is not present in the list.

Step 3:#### The user is assuming that the store might sell TVs, which is not explicitly stated in the available product information.

Step 4:#### Based on the available product information, the store only sells computers and laptops, and there is no mention of TVs.

Response to user:#### Hi there, thank you for reaching out. We currently specialize in computers and laptops, and do not have TVs in our product lineup. If you're looking for a new computer or laptop, I'd be happy to help you find the perfect one from our available options, such as the TechPro Ultrabook or the BlueWave Gaming Laptop. Let me know if you have any questions or need further assistance.


| **Line of Code**                                                      | **Purpose**                                                                                                 | **Input**                     | **Output / Result**                                    |
| --------------------------------------------------------------------- | ----------------------------------------------------------------------------------------------------------- | ----------------------------- | ------------------------------------------------------ |
| `user_message = f""" do you sell tvs"""`                              | Stores the customer's query in a variable.                                                                  | `"do you sell tvs"`           | `user_message` contains the customer's question.       |
| `messages = [`                                                        | Creates the conversation that will be sent to the LLM.                                                      | System message + User message | A list of conversation messages.                       |
| `{'role':'system', 'content': system_message}`                        | Gives instructions to the AI about its role and how to classify the query.                                  | `system_message`              | AI understands the task before reading the user query. |
| `{'role':'user', 'content': f"{delimiter}{user_message}{delimiter}"}` | Sends the user's question to the AI. The delimiter (`####`) separates the user input from the instructions. | `user_message`                | User query becomes `####do you sell tvs####`.          |
| `response = get_completion_from_messages(messages)`                   | Sends the complete conversation to the Groq/OpenAI model.                                                   | `messages`                    | AI analyzes the query and generates a response.        |
| `print(response)`                                                     | Displays the AI's response.                                                                                 | `response`                    | Prints the classification result on the screen.        |


===

| **Line of Code**                                                      | **Purpose**                                                                                                 | **Input**                     | **Output / Result**                                    |
| --------------------------------------------------------------------- | ----------------------------------------------------------------------------------------------------------- | ----------------------------- | ------------------------------------------------------ |
| `user_message = f""" do you sell tvs"""`                              | Stores the customer's query in a variable.                                                                  | `"do you sell tvs"`           | `user_message` contains the customer's question.       |
| `messages = [`                                                        | Creates the conversation that will be sent to the LLM.                                                      | System message + User message | A list of conversation messages.                       |
| `{'role':'system', 'content': system_message}`                        | Gives instructions to the AI about its role and how to classify the query.                                  | `system_message`              | AI understands the task before reading the user query. |
| `{'role':'user', 'content': f"{delimiter}{user_message}{delimiter}"}` | Sends the user's question to the AI. The delimiter (`####`) separates the user input from the instructions. | `user_message`                | User query becomes `####do you sell tvs####`.          |
| `response = get_completion_from_messages(messages)`                   | Sends the complete conversation to the Groq/OpenAI model.                                                   | `messages`                    | AI analyzes the query and generates a response.        |
| `print(response)`                                                     | Displays the AI's response.                                                                                 | `response`                    | Prints the classification result on the screen.        |


===
| **Step** | **Action**                                 | **Example**                              |
| -------- | ------------------------------------------ | ---------------------------------------- |
| 1        | Customer asks a question                   | `Do you sell TVs?`                       |
| 2        | Store the question                         | `user_message`                           |
| 3        | Combine system instructions and user query | `messages`                               |
| 4        | Send `messages` to the LLM                 | `get_completion_from_messages(messages)` |
| 5        | AI analyzes the question                   | Identifies it as a product inquiry.      |
| 6        | AI returns the category                    | General Inquiry → Product Information    |
| 7        | Print the response                         | Displays the JSON result.                |

====

Inner Monologue

Since we asked the LLM to separate its reasoning steps by a delimiter, we can hide the chain-of-thought reasoning from the final output that the user sees.

In [6]:
try:
    final_response = response.split(delimiter)[-1].strip()
except Exception as e:
    final_response = "Sorry, I'm having trouble right now, please try asking another question."

print(final_response)

Hi there, thank you for reaching out. We currently specialize in computers and laptops, and do not have TVs in our product lineup. If you're looking for a new computer or laptop, I'd be happy to help you find the perfect one from our available options, such as the TechPro Ultrabook or the BlueWave Gaming Laptop. Let me know if you have any questions or need further assistance.


Overall workflow

Customer Question
        │
        ▼
user_message
        │
        ▼
messages (System + User)
        │
        ▼
get_completion_from_messages()
        │
        ▼
Groq / LLM Model
        │
        ▼
Analyze the Query
        │
        ▼
Identify Category
        │
        ▼
Return JSON Response
        │
        ▼
print(response)


====


This pattern is commonly used in AI applications to safely extract the user-facing portion of an LLM response while providing a graceful fallback if something unexpected happens.